In [ ]:
first_train = True
first_valid = True

train_rows = 0
valid_rows = 0

for i, chunk in enumerate(pd.read_csv(INPUT_PATH, chunksize=100_000)):
    train_chunk = chunk[chunk["text_A"].isin(train_A)]
    valid_chunk = chunk[chunk["text_A"].isin(valid_A)]

    if not train_chunk.empty:
        train_chunk.to_csv(
            TRAIN_PATH,
            mode="w" if first_train else "a",
            header=first_train,
            index=False
        )
        first_train = False
        train_rows += len(train_chunk)

    if not valid_chunk.empty:
        valid_chunk.to_csv(
            VALID_PATH,
            mode="w" if first_valid else "a",
            header=first_valid,
            index=False
        )
        first_valid = False
        valid_rows += len(valid_chunk)

    # 🔔 진행 로그 (10 chunk마다)
    if (i + 1) % 10 == 0:
        print(
            f"[chunk {i+1}] "
            f"train rows: {train_rows:,} | "
            f"valid rows: {valid_rows:,}"
        )

print("✅ split & save finished")


[chunk 10] train rows: 899,240 | valid rows: 100,760
[chunk 20] train rows: 1,799,048 | valid rows: 200,952
[chunk 30] train rows: 2,698,599 | valid rows: 301,401
✅ split & save finished


In [ ]:
train_A, valid_A = train_test_split(
    A_list,
    test_size=0.1,
    random_state=42
)

train_A = set(train_A)
valid_A = set(valid_A)

print("text_A overlap:", len(train_A & valid_A))  # 반드시 0


text_A overlap: 0


In [ ]:
A_set = set()

for chunk in pd.read_csv(
    INPUT_PATH,
    chunksize=100_000,
    usecols=["text_A"]
):
    A_set.update(chunk["text_A"].dropna().unique())

A_list = list(A_set)
print("unique text_A:", len(A_list))


unique text_A: 410451


In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

BASE = "/content/drive/MyDrive/medical/capstone_work"

INPUT_PATH = f"{BASE}/training_data_clean_dedup_balanced.csv"
TRAIN_PATH = f"{BASE}/train.csv"
VALID_PATH = f"{BASE}/valid.csv"


01_train_valid_split_and_save.ipynb

목적:
누수 0 보장 split → 실제 파일로 저장

🔹 핵심 원칙

split 기준: text_A

train / valid 간:

text_A 겹치면 ❌

(text_A, text_B) 겹치면 ❌

RAM-safe (chunk 기반)

🔹 구현 전략

전체 CSV에서 고유 text_A 수집

text_A 기준으로 train / valid split

chunk 단위로 읽으면서

train_A에 속하면 → train.csv

valid_A에 속하면 → valid.csv

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

PATH = "/content/drive/MyDrive/medical/Capstone/training_data_clean_dedup_balanced.csv"
OUT_TRAIN = "/content/drive/MyDrive/medical/Capstone/train.csv"
OUT_VALID = "/content/drive/MyDrive/medical/Capstone/valid.csv"

# 1️⃣ text_A 수집
A_set = set()
for chunk in pd.read_csv(PATH, chunksize=100_000
, usecols=["text_A"]):
    A_set.update(chunk["text_A"].dropna().unique())

A_list = list(A_set)

# 2️⃣ text_A 기준 split
train_A, valid_A = train_test_split(
    A_list, test_size=0.1, random_state=42
)
train_A = set(train_A)
valid_A = set(valid_A)

print("text_A overlap:", len(train_A & valid_A))  # 0 이어야 함

# 3️⃣ 실제 파일 저장
first_train, first_valid = True, True

for chunk in pd.read_csv(PATH, chunksize=50000):
    train_chunk = chunk[chunk["text_A"].isin(train_A)]
    valid_chunk = chunk[chunk["text_A"].isin(valid_A)]

    if not train_chunk.empty:
        train_chunk.to_csv(
            OUT_TRAIN, mode="w" if first_train else "a",
            header=first_train, index=False
        )
        first_train = False

    if not valid_chunk.empty:
        valid_chunk.to_csv(
            OUT_VALID, mode="w" if first_valid else "a",
            header=first_valid, index=False
        )
        first_valid = False

print("✅ train / valid CSV 저장 완료")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
